In [ ]:
import tifffile as tf
import matplotlib.pyplot as plt
import skimage
import numpy as np
import pickle 
import cv2
import pandas as pd
import seaborn as sns 
from scipy.ndimage import gaussian_filter
from pyFlowSOM import map_data_to_nodes, som
from alpineer import io_utils, misc_utils
from pyarrow.lib import ArrowInvalid
from sklearn.preprocessing import MinMaxScaler
from pathlib import Path

In [ ]:
def compute_som_weights(pixels_subset_no_meta, prefix, xdim = 10, ydim = 10, num_passes = 1, lr_start=0.05, lr_end=0.01, seed = 42):
    som_weights = som(
            data=pixels_subset_no_meta.values.astype(np.float64), xdim=xdim, ydim=ydim,
            rlen=num_passes, alpha_range=(lr_start, lr_end), seed=seed
        )

    som_weights = np.reshape(som_weights, (xdim * ydim, som_weights.shape[-1]))
    weights_df = pd.DataFrame(som_weights, columns=pixels_subset_no_meta.columns.values)
    #weights_df.to_csv(pixel_clustering_path + '/' + prefix + '_som_weights.csv')
    return weights_df

In [ ]:
def assign_som_clusters(weights_df, pixels_no_meta, batch_size): 
    cluster_labels = []
    print(pixels_no_meta.shape[0])
    
    for i in np.arange(0, pixels_no_meta.shape[0], batch_size):
        
        print(i)
        cluster_labels.append(map_data_to_nodes(
            weights_df.values.astype(np.float64),
            pixels_no_meta.loc[
                i:min(i + batch_size-1, pixels_no_meta.shape[0]), pixels_no_meta.columns.values
            ].values.astype(np.float64)
        )[0])
    print(np.concatenate(cluster_labels).shape)
    pixels_no_meta['pixel_som_cluster'] = np.concatenate(cluster_labels)
    return pixels_no_meta
    
    

In [ ]:
def cluster(pixels_subset, pixels, prefix, batch_size, drop_position, weights_df = None, xdim = None, ydim= None):
    if(drop_position):
        pixels_subset_no_meta = pixels_subset.drop(columns=['X', 'Y'])
        pixels_no_meta = pixels.drop(columns=['X', 'Y'])
    else:
        pixels_subset_no_meta = pixels_subset
        pixels_no_meta = pixels
    print('computing som weights')
    if(weights_df is None):
        if(xdim is None and ydim is None):
            weights_df = compute_som_weights(pixels_subset_no_meta, prefix)
        else:
            weights_df = compute_som_weights(pixels_subset_no_meta, prefix, xdim=xdim, ydim=ydim)
        print(weights_df.shape)
    print('assigning som clusters')
    pixels_no_meta_som_clusters = assign_som_clusters(weights_df, pixels_no_meta, batch_size)
   

    return pixels_no_meta_som_clusters, weights_df

In [ ]:
def create_clustered_pixel(x, y, labels):
    x_max = np.max(x)
    y_max = np.max(y)
    img_labeled = np.zeros((x_max+1, y_max+1)) * np.nan
    img_labeled[x, y] = labels
    return img_labeled 

In [ ]:
import distinctipy
def create_color_dict(labels_unique, excluded_colors=None):
    color_arr = distinctipy.get_colors(len(labels_unique), exclude_colors=excluded_colors)
    cell_type_color_dict = {}
    for i in range(len(labels_unique)):
        cell_type_color_dict[labels_unique[i]] = color_arr[i]
    return cell_type_color_dict

In [ ]:
key = '/Multiplex 2/ANCA 9'

In [ ]:
registration_path = 'Y:/coskun-lab/Aditi/01/Results/registration/' + key + '/1x'

In [ ]:
color_deconvolution_path = 'Y:/coskun-lab/Aditi/01/Results/color_deconvolution' + key

In [ ]:
cell_clustering_path = 'Y:/coskun-lab/Aditi/01/Results/cell_segmentation_clustering/' + key + '/1x'

In [ ]:
fibrosis_path = 'Y:/coskun-lab/Aditi/01/Results/fibrosis_segmentation/' + key 

In [ ]:
Path.mkdir(Path('/'.join(fibrosis_path.split('/')[:-1])), exist_ok=True)

In [ ]:
Path.mkdir(Path(fibrosis_path), exist_ok=True)

In [ ]:
trichrome_blue = tf.imread(color_deconvolution_path + '/trichrome_blue_non_wsi.tif')

In [ ]:
with open(cell_clustering_path + '/segmentation_mask_merged_within_contour_for_pixel_clustering.pickle', 'rb') as f:
    cell_segmentation_mask = pickle.load(f)

In [ ]:
with open(registration_path + '/roi_coord_dict.pickle', 'rb') as f: 
    roi = pickle.load(f)

In [ ]:
cell_cluster_location_df = pd.read_csv(cell_clustering_path + '/cell_cluster_location_merged_unclassified_within_contour.csv').drop(columns=['Unnamed: 0'])

In [ ]:
Trichrome = tf.imread(registration_path + '/Trichrome_registered.tif')

In [ ]:
tile_location_list = []
for i in range(0, cell_segmentation_mask.shape[0], 512):
    for j in range(0, cell_segmentation_mask.shape[1], 512):
        tile_location_list.append((i, j))

In [ ]:
cell_segmentation_mask_binary = np.zeros(cell_segmentation_mask.shape)
cell_segmentation_mask_binary[cell_segmentation_mask!=0] = 1
cell_segmentation_mask_binary = cv2.dilate(cv2.erode(cell_segmentation_mask_binary, np.ones((3, 3), np.uint8), iterations=1), np.ones((103, 103), np.uint8), iterations=1)

contours, _ = cv2.findContours(cell_segmentation_mask_binary.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contour = max(contours, key=cv2.contourArea)

contour_drawn = np.zeros((cell_segmentation_mask_binary.shape[0], cell_segmentation_mask_binary.shape[1]))
contour_drawn = cv2.drawContours(contour_drawn, [contour], -1, (255, 255, 255), -1)


segmentation_mask_offset = np.zeros(cell_segmentation_mask.shape)

offset = 0
for l in range(len(tile_location_list)):
    i, j = tile_location_list[l]
    segmented_tile = cell_segmentation_mask[i:i+512, j:j+512] 
    segmented_tile_non_background_mask = segmented_tile!=0 
    segmented_tile_offset = segmented_tile.copy()
    segmented_tile_offset[segmented_tile_non_background_mask] = segmented_tile[segmented_tile_non_background_mask] + offset
    segmentation_mask_offset[i:i+512, j:j+512] = segmented_tile_offset
    offset+=np.max(segmented_tile)

segmentation_mask_processed = segmentation_mask_offset.copy()
segmentation_mask_processed[contour_drawn == 0] = 0

In [ ]:
segmentation_mask_processed_binary = segmentation_mask_processed.copy()
segmentation_mask_processed_binary[segmentation_mask_processed_binary!=0] = 255

In [ ]:
base_map = np.zeros(segmentation_mask_processed.shape)
for x, y in zip(list(cell_cluster_location_df['X']), list(cell_cluster_location_df['Y'])):
    base_map[y, x] += 1 

density_sigma = 15
density_map = gaussian_filter(base_map, sigma=density_sigma, mode='constant')

In [ ]:
trichrome_blue = 255 - trichrome_blue
trichrome_blue = (trichrome_blue - trichrome_blue.min()) / (trichrome_blue.max() - trichrome_blue.min())

In [ ]:
trichrome_sigma = 15
trichrome_blue_map = gaussian_filter(trichrome_blue, sigma=trichrome_sigma, mode='constant')

In [ ]:
fig, ax = plt.subplots(len(roi), 3)
i = 0
fig.set_figheight(20)
fig.set_figwidth(20)
for name, coord in roi.items():
    ax[i, 0].imshow(trichrome_blue_map[coord[0]:coord[1], coord[2]:coord[3]], cmap='jet')
    ax[i, 1].imshow(density_map[coord[0]:coord[1], coord[2]:coord[3]], cmap='jet')
    ax[i, 2].imshow(Trichrome[coord[0]:coord[1], coord[2]:coord[3]])

    ax[i, 0].axis('off')
    ax[i, 1].axis('off')
    ax[i, 2].axis('off')
    
    i+=1


In [ ]:
with open(fibrosis_path + '/trichrome_blue_map_repeat.pickle', 'wb') as f:
    pickle.dump(trichrome_blue_map, f)

In [ ]:
with open(fibrosis_path + '/cell_density_map_repeat.pickle', 'wb') as f:
    pickle.dump(density_map, f)

In [ ]:
cell_centroid_data = np.stack([cell_cluster_location_df['X'], cell_cluster_location_df['Y']]).T

In [ ]:
fibrosis_density_stack = np.stack([trichrome_blue_map, density_map])
num_images, height, width = fibrosis_density_stack.shape
num_pixels = height * width
fibrosis_density_stack_flattened = fibrosis_density_stack.reshape(num_images, num_pixels).T
fibrosis_density_df = pd.DataFrame(fibrosis_density_stack_flattened, columns=['Fibrosis', 'Cell Density'])

In [ ]:
x = fibrosis_density_stack.shape[1]
y = fibrosis_density_stack.shape[2]
        
pixel_range = [i for i in range(x)]
fibrosis_density_df['X'] = np.repeat(pixel_range, y)
fibrosis_density_df['Y'] = [i for i in range(y)] * x

In [ ]:
trichrome_blue_for_non_empty = gaussian_filter(trichrome_blue, sigma=5, mode='constant')

In [ ]:
thr = 0.15

In [ ]:
fig, ax = plt.subplots(len(roi), 2)
i = 0
fig.set_figheight(20)
fig.set_figwidth(10)
for name, coord in roi.items():
    ax[i, 0].imshow(trichrome_blue_for_non_empty[coord[0]:coord[1], coord[2]:coord[3]]>thr)
    ax[i, 1].imshow(Trichrome[coord[0]:coord[1], coord[2]:coord[3]])
    
    i+=1

In [ ]:
trichrome_blue_non_empty = trichrome_blue_for_non_empty > thr
trichrome_blue_flattened = trichrome_blue_non_empty.reshape(trichrome_blue_non_empty.shape[0]*trichrome_blue_non_empty.shape[1])

In [ ]:
trichrome_blue_non_empty_pixels = pd.DataFrame(trichrome_blue_flattened, columns=['Non-empty'])

In [ ]:
fibrosis_density_df['Non-empty'] = trichrome_blue_non_empty_pixels['Non-empty']
fibrosis_density_df['Contour'] = contour_drawn.astype(bool).flatten()
fibrosis_density_non_empty_pixels = fibrosis_density_df[(fibrosis_density_df['Non-empty'] == True) & (fibrosis_density_df['Contour'] == True)]

In [ ]:
fibrosis_density_non_empty_pixels = fibrosis_density_non_empty_pixels.drop(columns=['Non-empty', 'Contour'])

In [ ]:
x = fibrosis_density_non_empty_pixels['X']
y = fibrosis_density_non_empty_pixels['Y']
non_empty_pixels_no_meta = fibrosis_density_non_empty_pixels.drop(columns=['X', 'Y'])
scaler = MinMaxScaler()
x_scaled = scaler.fit_transform(non_empty_pixels_no_meta)
fibrosis_density_non_empty_pixels_scaled = pd.DataFrame(x_scaled, columns=non_empty_pixels_no_meta.columns)
fibrosis_density_non_empty_pixels_scaled['X'] = list(x)
fibrosis_density_non_empty_pixels_scaled['Y'] = list(y)

In [ ]:
pixels_no_meta_som_clusters, weights_df = cluster(fibrosis_density_non_empty_pixels_scaled, fibrosis_density_non_empty_pixels_scaled, 'fibrosis_inflammation_edema', 100000000, drop_position=True, weights_df = None, xdim = 2, ydim= 2)

In [ ]:
non_empty_pixels_som_clusters = pixels_no_meta_som_clusters.copy()
non_empty_pixels_som_clusters['X'] = list(fibrosis_density_non_empty_pixels_scaled['X'])
non_empty_pixels_som_clusters['Y'] = list(fibrosis_density_non_empty_pixels_scaled['Y'])

In [ ]:
non_empty_pixels_som_clusters.to_csv(fibrosis_path + '/fibrosis_density_som_clusters_repeat.csv')

In [ ]:
x = non_empty_pixels_som_clusters.X.tolist()
y = non_empty_pixels_som_clusters.Y.tolist()
wsi_labeled = create_clustered_pixel(x, y, np.array(list(non_empty_pixels_som_clusters.pixel_som_cluster)))
wsi_labeled[wsi_labeled == 0] = np.nan

In [ ]:
with open(fibrosis_path + '/wsi_labeled.pickle', 'wb') as f:
    pickle.dump(wsi_labeled, f)

In [ ]:
cluster_color_dict = create_color_dict(list(set(list(non_empty_pixels_som_clusters['pixel_som_cluster'].unique()))))

In [ ]:
with open(fibrosis_path + '/cluster_color_dict.pickle', 'wb') as f:
    pickle.dump(cluster_color_dict, f)

In [ ]:
fig, ax = plt.subplots(1)
fig.set_figheight(150)
fig.set_figwidth(150)

for name, coord in roi.items():
    wsi_labeled = cv2.rectangle(wsi_labeled, (coord[2], coord[0]), (coord[3], coord[1]), np.nan, 50)
cmap = plt.cm.colors.ListedColormap(list(cluster_color_dict.values()))
ax.imshow(wsi_labeled, cmap=cmap)
ax.axis('off')
plt.savefig(fibrosis_path + '/wsi_clusters_repeat.png')

In [ ]:
fig, ax = plt.subplots(2, len(list(roi.keys())))
fig.set_figheight(30)
fig.set_figwidth(50)
i = 0
j = 0
for name, coord in roi.items():
    print(name)
    img_labeled = wsi_labeled[coord[0]:coord[1], coord[2]:coord[3]]
    ax[0, j].imshow(img_labeled, cmap=plt.cm.colors.ListedColormap(list(cluster_color_dict.values())), vmin=1, vmax=4)
    ax[0, j].axis('off')
    ax[1, j].imshow(Trichrome[coord[0]:coord[1], coord[2]:coord[3]])
    ax[1, j].axis('off')
    j+=1 

    plt.subplots_adjust(wspace=0.025, hspace=0.025)

plt.savefig(fibrosis_path + '/regions_clusters_repeat.png')

In [ ]:
import matplotlib.patches as mpatches
fig, ax = plt.subplots(1)

i = 0
patches = [
    mpatches.Patch(color=color, label=label)
    for label, color in zip(list(cluster_color_dict.keys()), list(cluster_color_dict.values()))]
ax.legend(patches, list(cluster_color_dict.keys()), loc='center', frameon=False, ncol=int(len(list(cluster_color_dict.keys()))))
ax.axis('off')
i+=1
plt.show()

In [ ]:
sns.set(font_scale=1) 
sns.clustermap(non_empty_pixels_som_clusters.groupby('pixel_som_cluster').median().drop(columns=['X', 'Y']).transpose(), figsize=(5, 2), yticklabels=True, row_cluster=False, cbar_kws={'location': 'right'})
plt.savefig(fibrosis_path + '/clusters_heatmap.png')

In [ ]:
non_empty_pixels_som_clusters.groupby('pixel_som_cluster').median()